# Loyiha 3: Linear Regression to'liq — polynomial, learning curves, Ridge/Lasso — Wage (ISLR, AQSh 2003–2009)

**Talaba:** student3 · **Modul:** M3 (Linear Regression to'liq) + M2 · **Kitob (ML_GUIDE):** 4-bob, 5-bob, 7-bob (Regularization) · **Qiyinlik:** ⭐⭐⭐⭐ · **Taxminiy vaqt:** 8–10 soat · **Muddat:** ____________
> **Qoidalar.** Bu notebook — sizning ish daftaringiz. Har bir `# TODO` katakni to'ldiring, har bir **Savol** ga darhol pastida yangi *markdown* katak ochib 2–5 gap bilan javob bering ("ha/yo'q" baholanmaydi). Topshirishdan oldin `Kernel → Restart & Run All` — notebook yuqoridan pastga **xatosiz** ishlashi shart. Internetdan o'rganish — mumkin va kerak; tayyor kodni nusxalash — yo'q (himoyada har bir qatorni tushuntirib bera olishingiz kerak).

## Loyiha haqida

Haqiqiy ma'lumot kamdan-kam to'g'ri chiziqqa tushadi. Bu loyihada to'liq **professional regressiya tahlili** qilasiz: EDA → baseline → chiziqli model → polynomial (underfit/overfit) → learning curves → Ridge/Lasso/ElasticNet bilan alpha ni cross-validation orqali tanlash → yakuniy jadval va xulosa.

**Nimani o'rganasiz:** residual tahlili · `Pipeline` bilan leakage-siz scaling · degree vs xato grafigi · bias/variance ni learning curve dan o'qish · alpha ning koeffitsiyentlarga ta'siri · Lasso feature selection · `cross_val_score` (🔍 yangi).

## Dataset: Wage (ISLR, AQSh 2003–2009)

3000 erkak ishchi (AQSh Mid-Atlantic, 2003–2009): yosh, ta'lim, oilaviy holat, ish turi, salomatlik → **soatlik ish haqi** ($). *Introduction to Statistical Learning* kitobining 1-bobidagi dataset. Kuchli **nochiziqli** (yosh ↔ maosh — teskari U) va ko'p **kategorik** ustunli — chiziqli model uchun qiyin, lekin ibratli.

**Manba:** https://raw.githubusercontent.com/JWarmenhoven/ISLR-python/master/Notebooks/Data/Wage.csv (asl: ISLR `Wage`)

| Ustun | Ma'nosi |
|---|---|
| year, age | yil (2003–2009), yosh (18–80) |
| maritl, race, education, jobclass, health, health_ins | kategorik: oilaviy holat (5), irq (4), ta'lim (5 daraja), Industrial/Information, salomatlik, sug'urta |
| sex, region | **doimiy** (faqat Male, faqat Middle Atlantic) — foydasiz |
| logwage | log(wage) — **LEAKAGE**, olib tashlang |
| **wage** | soatlik maosh, $ — target (20–318, median 105) |

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
url = "https://raw.githubusercontent.com/JWarmenhoven/ISLR-python/master/Notebooks/Data/Wage.csv"
df = pd.read_csv(url, index_col=0)             # 3000 ishchi
TARGET = 'wage'
print(df.shape); df.head()

## 1-vazifa: EDA va tozalash
`describe()`, missing values, target histogrammasi, korrelyatsiya heatmap (`sns.heatmap`), target vs 3 ta eng kuchli feature scatter. Har bir grafik ostiga **1 gap kuzatuv** yozing.

**Tozalash:** `logwage` = log(target) — leakage, olib tashlang. `sex` va `region` — bitta qiymat, olib tashlang. Qolgan 6 ta kategorik ustunni `pd.get_dummies(drop_first=True, dtype=float)`. `education` **tartibli** (1 < 2 < … < 5) — one-hot o'rniga `str[0].astype(int)` bilan raqam qilib ham sinang (ikkalasini solishtiring).

**Savol 1.** Qaysi feature'lar target bilan **nochiziqli** bog'langan ko'rinadi (scatter da egri)? Qaysi juft feature'lar o'zaro kuchli korrelyatsiyalangan (|r| > 0.7) va bu chiziqli modelga qanday ta'sir qilishi mumkin?

In [ ]:
df = df.drop(columns=['logwage', 'sex', 'region'])
# TODO: get_dummies (6 kategorik), X = ..., y = df[TARGET].values
# TODO: EDA grafiklari

## 2-vazifa: Baseline va chiziqli model
Split (80/20, `random_state=42`). **Baseline**: hamma uchun `y_train.mean()` → test MAE/RMSE. Keyin `make_pipeline(StandardScaler(), LinearRegression())` → test MAE/RMSE/R². Residual grafigi: x = `y_pred`, y = `y_test − y_pred`.

**Savol 2.** Residual grafigida naqsh (pattern) bormi — egri, "karnay" (variance o'sadi), klasterlar? Ideal residual grafigi qanday ko'rinishi kerak va nega (kitob 4-bob "Assumptions")?

In [ ]:
# TODO: baseline, pipeline LinearRegression, residual plot

## 3-vazifa: Polynomial regression — degree tajribasi
`degree ∈ {1, 2, 3, 4, 5}`: `make_pipeline(StandardScaler(), PolynomialFeatures(d), StandardScaler(), LinearRegression())`. Har degree uchun **train** va **test** RMSE. Bitta grafik: x = degree, ikki chiziq. Har degree da feature soni nechta bo'ldi (`n_features_out`)?

**Savol 3.** Qaysi degree dan boshlab overfitting boshlanadi (train ↓, test ↑)? Nega `PolynomialFeatures` dan keyin **yana** `StandardScaler` qo'ydik? Feature soni degree bilan qanday o'sadi (formula: C(n+d, d))?

In [ ]:
# TODO: degree 1..5, train/test RMSE grafigi, feature soni

## 4-vazifa: Learning curves
`learning_curve(..., train_sizes=np.linspace(0.1, 1.0, 10), cv=5, scoring='neg_root_mean_squared_error', shuffle=True, random_state=42)` — 3 ta model uchun: degree=1, eng yaxshi degree, overfit degree. 3 ta subplot.

**Savol 4.** Har bir egri chiziqni bias/variance tilida izohlang (M3 4-qism "uchta shakl"). Qaysi modelga **ko'proq ma'lumot** yordam beradi, qaysiga yordam bermaydi? `neg_` prefiksi nega (M3 dagi tuzoq)?

In [ ]:
# TODO: 3 ta learning curve

## 5-vazifa: Ridge va Lasso — alpha ni cross-validation bilan tanlash
Eng yaxshi degree (yoki overfit degree — qiziqroq!) bilan: `alphas = np.logspace(-3, 3, 13)`. Har alpha uchun `cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')` o'rtachasi. Grafik: x = alpha (`plt.xscale('log')`), Ridge va Lasso ikki chiziq. Eng yaxshi alpha larni toping.

**Savol 5.** 🔍 `cross_val_score` nima qiladi (5-fold)? Nega alpha ni **test** to'plamida emas, CV bilan tanlaymiz? Alpha juda katta bo'lsa nima bo'ladi (kitob 7-bob "kuch tugmasi")?

In [ ]:
# TODO: alpha grid, cross_val_score, grafik, eng yaxshi alpha

## 6-vazifa: Koeffitsiyent yo'llari va Lasso feature selection
Har alpha uchun Ridge va Lasso koeffitsiyentlarini saqlang → **coefficient path** grafigi (x = alpha log, y = har bir koeffitsiyent chizig'i). Eng yaxshi alpha da Lasso nechta koeffitsiyentni **aynan 0** qildi? Qaysilarini?

**Savol 6.** Nega Lasso nolga tushiradi, Ridge esa faqat kichraytiradi — L1 vs L2 geometriyasi (kitob 7-bob "L1 vs L2")? Nolga tushgan feature'lar 1-vazifadagi kuzatuvlaringizga mos keladimi?

In [ ]:
# TODO: coefficient path (Ridge, Lasso), nol koeffitsiyentlar

## 7-vazifa: ElasticNet va yakuniy jadval
`ElasticNet(l1_ratio ∈ {0.2, 0.5, 0.8})` ni ham alpha grid bilan sinang. Yakuniy jadval (test): Baseline · Linear · Poly(best) · Ridge · Lasso · ElasticNet — MAE, RMSE, R². Eng yaxshi modelni tanlang va **5 gap** asoslang.

**Target taqsimoti.** `wage` histogrammasida ikki cho'qqi: asosiy massa 50–150 va alohida guruh 250+ (79 kishi — "yuqori maosh"). Bu guruh chiziqli modelning residual larida qanday ko'rinadi? `np.log(wage)` ni target qilib qayta o'rgating va bashoratni `np.exp` bilan $ ga qaytarib RMSE ni solishtiring. Log yordam berdimi? Nega ISLR mualliflari `logwage` ustunini qo'shgan?

**Savol 7.** Eng yaxshi model baseline dan necha % yaxshi? Agar ish beruvchi "oddiy, tushuntirib beriladigan model kerak" desa, qaysini tavsiya qilasiz va nima yo'qotasiz?

In [ ]:
# TODO: ElasticNet, yakuniy jadval

## 🔍 Tadqiqot (majburiy — kamida 2 tasi)
1. `RidgeCV` va `LassoCV` — 5-vazifani ular bilan takrorlang. Natija bir xilmi? `LassoCV` qaysi yo'l bilan alpha tanlaydi (hujjat: `alphas`, `cv`, `n_alphas`)?
2. **Residual normalligi**: eng yaxshi model residual lari uchun `scipy.stats.probplot` (QQ-plot) chizing. Normal taqsimotdan qayerda chetlanadi va bu nimani anglatadi?
3. **Multicollinearity — VIF**: `statsmodels.stats.outliers_influence.variance_inflation_factor` bilan har feature VIF ini hisoblang. VIF > 10 bo'lganlar qaysilar? Ridge ularga qanday yordam beradi?
4. ISLR 1-bobidagi 3 ta grafikni takrorlang: `wage` vs `age`, `wage` vs `year`, `wage` vs `education` (boxplot). Faqat `age` bilan degree 1 / 3 / 5 polynomial — R² 0.03 → 0.07: yosh maoshni **deyarli** tushuntirmaydi, lekin egri chiziq shakli aniq — bu qanday bo'lishi mumkin (R² kichik ≠ bog'liqlik yo'q)? `education` yolg'iz R² qancha?

In [ ]:
# TODO: tadqiqot

## ⭐ Bonus (+10)
`GridSearchCV` bilan **bitta** pipeline ichida `polynomialfeatures__degree` × `ridge__alpha` ni birgalikda qidiring (`cv=5`). Eng yaxshi kombinatsiya 3–5-vazifadagi "qo'lda" tanlovingiz bilan mos keldimi? `cv_results_` dan heatmap chizing.

## Topshirish

- Fayl nomi: `student3_P3.ipynb` (shu fayl, to'ldirilgan). `Restart & Run All` qilingan, barcha grafiklar ko'rinib turibdi.
- Oxirgi katak — **Xulosa** (markdown, 6–10 gap): nimani o'rgandingiz, qaysi natija kutilmagan bo'ldi, nimani hali tushunmadingiz.
- Himoya: 5 daqiqa — istalgan katakdagi kodni tushuntirib berasiz.

## Baholash (100 ball + bonus)

| Mezon | Ball |
|---|---|
| 1–7 vazifalar (jadvallar, grafiklar, to'g'ri pipeline) | 40 |
| Savollarga tahliliy javoblar (7 ta) | 20 |
| Tadqiqot (2 ta) | 15 |
| Leakage yo'q: scaler/CV faqat train da, test bir marta | 10 |
| Grafik sifati + kod tozaligi | 15 |
| ⭐ Bonus | +10 |